# polygon → MOC → shard → 3-D → numpy

The whole zagg read stack in five functions, two `pip install`s, and zero
credentials. A geojson polygon becomes a morton MOC; the MOC checks itself
against the store's own coverage; the covered shards open with timings; one
shard renders in 3-D (ATL03 + GEDI together); the current view exports to a
numpy tensor and saves to disk. Everything below is reader-side — `mortie`
for the geometry, `moczarr` for the store — running anonymously against
public S3, binder-ready.

In [1]:
# %pip install mortie moczarr matplotlib ipympl ipywidgets

import json
import resource
import time

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LogNorm

import mortie
import moczarr
from moczarr.convention import point_to_area29
from moczarr.ragged import open_ragged, read_ragged

# The region pair: one ATL03 photon store, one GEDI waveform-flux store, on
# the same o9 shard grid. Flip REGION to "california" once the source-coop
# fleet rerun lands -- nothing else changes.
REGIONS = {
    "serc": {
        "atl03": ("s3://sliderule-public/zagg-demo/serc_tdigest_strata.zarr", "19/h_tdigest_signal"),
        "gedi": ("s3://sliderule-public/zagg-demo/serc_gedi_flux.zarr", "18/rx_flux"),
    },
    # "california": {
    #     "atl03": ("s3://us-west-2.opendata.source.coop/.../california_tdigest.zarr", "19/h_tdigest_signal"),
    #     "gedi": ("s3://us-west-2.opendata.source.coop/.../california_gedi_flux.zarr", "18/rx_flux"),
    # },
}
REGION = REGIONS["serc"]
S3 = {"region": "us-west-2"}  # add "anonymous": True on the public source-coop endpoint

rss = lambda: resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20  # noqa: E731

## One polygon in, covered shards out

In [3]:
def coverage(geojson, order=9):
    """Polygon -> MOC -> containment check against every store's own coverage
    -> the o9 shards to read. Raises if the polygon leaves the stores."""
    ring = np.asarray(geojson["features"][0]["geometry"]["coordinates"][0])
    moc = mortie.morton_coverage_moc(ring[:, 1], ring[:, 0], order=order)
    shards = None
    for name, (root, _field) in REGION.items():
        cov = moczarr.load_root_coverage(root, **S3)
        if not len(moczarr.root_coverage_and(cov, moc)):
            raise moczarr.NoCoverageError(f"polygon is outside the {name} store's coverage")
        st = moczarr.open_object_store(root, **S3)
        leaves = moczarr.candidate_leaves(root, moczarr.read_manifest(root, store=st), aoi=moc, store=st)
        ids = {leaf.rsplit("/", 1)[-1].split(".")[0] for leaf in leaves}
        shards = ids if shards is None else (shards & ids)
    print(f"{len(shards)} shard(s) cover the polygon in both stores")
    return sorted(shards)


aoi = {"features": [{"geometry": {"coordinates": [[  # a ~4 km box on the SERC tract
    [-76.56, 38.87], [-76.50, 38.87], [-76.50, 38.91], [-76.56, 38.91], [-76.56, 38.87]
]]}}]}  # or: aoi = json.load(open("area.geojson"))
shards = coverage(aoi)
shards

PermissionDeniedError: The operation lacked the necessary privileges to complete for path zagg-demo/serc_tdigest_strata.zarr/coverage.moc: Error performing GET https://s3.us-west-2.amazonaws.com/sliderule-public/zagg-demo/serc_tdigest_strata.zarr/coverage.moc in 122.867292ms - Server returned non-2xx status code: 403 Forbidden: <?xml version="1.0" encoding="UTF-8"?>
<Error><Code>AccessDenied</Code><Message>Access Denied</Message><RequestId>88ZFT4GG4AASBK7S</RequestId><HostId>Vp//rVnkt0ZMh20QHhUu5nv2BKg0E3SDuxX2mpol6Ch10NtaGMNBLaMcyhIngftEi2+OmCoOeGTwHZEBOFQzkJ3PSrNmRurZ</HostId></Error>

Debug source:
PermissionDenied {
    path: "zagg-demo/serc_tdigest_strata.zarr/coverage.moc",
    source: RetryError(
        RetryErrorImpl {
            method: GET,
            uri: Some(
                https://s3.us-west-2.amazonaws.com/sliderule-public/zagg-demo/serc_tdigest_strata.zarr/coverage.moc,
            ),
            retries: 0,
            max_retries: 10,
            elapsed: 122.8672

## Open one shard — every dataset, timed

In [ ]:
def open_shard(shard):
    """Open the shard's leaf in every store; print open+sweep timing and RSS."""
    handles = {}
    for name, (root, field) in REGION.items():
        t0, r0 = time.perf_counter(), rss()
        store = moczarr.open_leaf(root, shard, **S3)
        arr, element = open_ragged(store, field)
        n = sum(len(v) for _, v in read_ragged(store, field))
        print(f"{name:6s} open+sweep {time.perf_counter() - t0:5.1f}s "
              f"(+{rss() - r0:4.0f} MB) — {n:,} centroids, element {element}")
        handles[name] = (store, field)
    return handles


handles = open_shard(shards[0])

## The 3-D view — both sensors, exact centroids, time-aware

In [ ]:
from ipywidgets import Checkbox, Dropdown, HBox, VBox, interactive_output

_SIDE12 = float(np.sqrt(4 * np.pi / (12 * 4**12)) * 6_371_000)  # o12 block side, m


def _grid_xy(words, block_order=12):
    """Word -> (x, y) meters in its o12 block's lattice frame (vectorized
    spec-section-1 bit decode; a word keeps its own order's precision)."""
    w = np.asarray(words, dtype=np.uint64)
    s = (w & np.uint64(63)).astype(np.int64)
    o = np.where(s <= 27, s, np.where((s - 28) % 5 == 0, 28, 29)).astype(np.int64)
    x = np.zeros(len(w)); y = np.zeros(len(w))
    for L in range(block_order + 1, 30):
        m = np.flatnonzero(o >= L)
        if not len(m):
            continue
        rank = (((w[m] >> np.uint64(6 + 2 * (27 - L))) & np.uint64(3)).astype(np.int64)
                if L <= 27 else ((s[m] - 28) // 5 if L == 28 else (s[m] - 28) % 5 - 1))
        cx, cy = mortie.rank_to_xy(rank, 1)
        scale = 2.0 ** -(L - block_order)
        x[m] += np.asarray(cy, dtype=float) * scale
        y[m] += np.asarray(cx, dtype=float) * scale
    half = 0.5 * (2.0 ** -(o - block_order).astype(float))
    return (x + half) * _SIDE12, (y + half) * _SIDE12


def _load(store, field):
    """One sensor's centroids: z, weight, xy (located words when present,
    cell centers otherwise), acquisition days (times sibling when present)."""
    arr, _ = open_ragged(store, field)
    attrs = dict(arr.attrs)
    locname = (attrs.get("ragged") or {}).get("locations")
    tname = attrs.get("times")
    zs, wts, cells, locw, seq = [], [], [], [], []
    for row in read_ragged(store, field, locations=bool(locname)):
        seq.append(row[0])
        v = np.asarray(row[1])
        zs.append(v[:, 0]); wts.append(v[:, 1])
        cells.append(np.full(len(v), row[0], dtype=np.uint64))
        if locname:
            locw.append(np.asarray(row[2], dtype=np.uint64))
    z, wt = np.concatenate(zs), np.concatenate(wts)
    cells = np.concatenate(cells)
    sh = np.uint64(6 + 2 * (27 - 12))
    blocks = ((cells >> sh) << sh) | np.uint64(12)  # o12 ancestor by truncation
    x, y = _grid_xy(np.concatenate(locw) if locname else cells)
    t = None
    if tname:
        tmap = {int(r[0]): np.asarray(r[1], dtype=np.uint64).ravel()
                for r in read_ragged(store, field.rsplit("/", 1)[0] + "/" + tname)}
        tw = np.concatenate([tmap[int(c)] for c in seq])
        ns2018 = float((np.datetime64("2018-01-01") - np.datetime64("1850-01-01"))
                       // np.timedelta64(1, "ns"))
        t = (np.asarray(mortie.toc2time(tw)[0], dtype="float64") - ns2018) / 86.4e12
    return {"z": z, "wt": wt, "x": x, "y": y, "blocks": blocks, "t": t}


class View:
    """Holds what is on screen so `export` knows which block you mean."""
    shard = None
    block = None


def view3d(handles, shard):
    view = View()
    view.shard = shard
    data = {name: _load(store, field) for name, (store, field) in handles.items()}
    joint = sorted(set.intersection(*(set(np.unique(d["blocks"]).tolist()) for d in data.values())))
    dd = Dropdown(options=[(moczarr.morton_decimal(w), w) for w in joint], description="block")
    tc = Checkbox(value=False, description="color by time")

    def draw(block, by_time):
        view.block = block
        fig = plt.figure(figsize=(11, 5))
        for k, (name, d) in enumerate(data.items()):
            m = d["blocks"] == np.uint64(block)
            ax = fig.add_subplot(1, 2, k + 1, projection="3d")
            alpha = np.clip(d["wt"][m] / max(np.percentile(d["wt"][m], 98), 1e-9), 0.08, 1)
            if by_time and d["t"] is not None:
                pts = ax.scatter(d["x"][m], d["y"][m], d["z"][m], c=d["t"][m], s=1.5,
                                 cmap="turbo", alpha=alpha)
                fig.colorbar(pts, shrink=0.5, label="days since 2018-01-01")
            else:
                pts = ax.scatter(d["x"][m], d["y"][m], d["z"][m], c=d["wt"][m], s=1.5,
                                 cmap="viridis", norm=LogNorm(), alpha=alpha)
                fig.colorbar(pts, shrink=0.5, label="weight")
            ax.set_title(name, fontsize=10)
            ax.set_zlabel("elevation (m)")
        fig.suptitle(f"shard {shard} — block {moczarr.morton_decimal(block)}")
        plt.show()

    display(VBox([HBox([dd, tc]), interactive_output(draw, {"block": dd, "by_time": tc})]))
    return view


view = view3d(handles, shards[0])

## Export what you see — a numpy tensor, shaped your way

In [ ]:
from moczarr.hhdc import read_tensors


def export(view, sensor, n_bins=64, resolution=1.0, fit="degrade_resolution"):
    """The block on screen -> (rows, cols, n_bins) numpy tensor + z metadata.
    `n_bins`/`resolution` set the vertical shape; the xy shape follows the
    sensor's cell order within the o12 block."""
    store, field = handles[sensor]
    t, mask, (z0, dz), w = next(
        b for b in read_tensors(store, field, n_bins=n_bins, resolution=resolution,
                                block_order=12, fit=fit)
        if int(b[3]) == int(view.block)
    )
    print(f"{sensor}: {t.shape} tensor, z = {z0:.1f} m + bin * {dz:g} m")
    return t, {"z0": z0, "dz": dz, "block": moczarr.morton_decimal(w)}


gedi, meta = export(view, "gedi", n_bins=128, resolution=0.5)
np.save(f"gedi_{meta['block']}.npy", gedi)

atl03, meta = export(view, "atl03")           # default 64 x 1 m bins
np.save(f"atl03_{meta['block']}.npy", atl03)

Five functions, two libraries, one polygon — coverage, shards, timings,
the paired 3-D view, and tensors on disk.